In [1]:
from pathlib import Path
from difflib import SequenceMatcher
import re
import unicodedata
import warnings

from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import numpy as np
import pandas as pd
import spacy
from IPython.display import display
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split

SEED = 12
K_VALUES = (5, 7, 8, 9, 10, 12)
N_JOBS = 4
NLP_PROCESSES = 1

# Valori iniziali da verificare sui campioni del punto 5.
MIN_KNOWN_TERMS = 3
MIN_TOPIC_MARGIN = 0.10

cwd = Path.cwd()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "01_assignment").is_dir()),
    cwd,
)
INPUT_FILE = repo_root / "01_assignment" / "output_additional_truths" / "truths_integrated.csv.gz"
OUTPUT_FILE = repo_root / "02_assignment" / "output" / "post_categories.csv"

In [2]:
truths = pd.read_csv(
    INPUT_FILE,
    usecols=["id", "text"],
    dtype={"id": "string", "text": "string"},
    keep_default_na=False,
    na_values={"text": [""]},
)
assert truths["id"].ne("").all() and truths["id"].is_unique

missing_text = truths["text"].isna()
blank_text = truths["text"].notna() & truths["text"].str.strip().eq("")
has_text = ~(missing_text | blank_text)

display(pd.Series({
    "Post originali": len(truths),
    "Testo mancante": int(missing_text.sum()),
    "Testo di soli spazi": int(blank_text.sum()),
    "Testo presente": int(has_text.sum()),
}, name="Post").to_frame())

,Post
Post originali,946376
Testo mancante,11767
Testo di soli spazi,0
Testo presente,934609


In [3]:
def remove_html(text):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)
        return BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)


posts = truths.loc[has_text, ["id", "text"]].drop_duplicates("text").reset_index(drop=True)
posts["clean_text"] = posts["text"].map(lambda text: unicodedata.normalize("NFKC", text)).str.lower()
posts["clean_text"] = (
    posts["clean_text"]
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"<emoji:[^>]*>", " ", regex=True)
    .map(remove_html)
    .str.replace(r'class="(?:u-url mention status-link|mention hashtag status-link)', " ", regex=True)
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"[^\w\s'’]|_", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

usable_text = posts["clean_text"].str.contains(r"[^\W\d_]", regex=True)
print(f"Testi originali distinti con testo presente: {len(posts):,}")
print(f"Testi distinti senza lettere dopo la pulizia: {(~usable_text).sum():,}")

Testi originali distinti con testo presente: 682,943
Testi distinti senza lettere dopo la pulizia: 54,814


In [4]:
# Expand contractions
# Dictionary of common English contractions and their expanded forms
contractions = {
    # Forms of "to be"
    r"\bi['’]m\b": "i am",
    r"\byou['’]re\b": "you are",
    r"\bhe['’]s\b": "he is",
    r"\bshe['’]s\b": "she is",
    r"\bit['’]s\b": "it is",
    r"\bwe['’]re\b": "we are",
    r"\bthey['’]re\b": "they are",

    # Forms of "will"
    r"\bi['’]ll\b": "i will",
    r"\byou['’]ll\b": "you will",
    r"\bhe['’]ll\b": "he will",
    r"\bshe['’]ll\b": "she will",
    r"\bit['’]ll\b": "it will",
    r"\bwe['’]ll\b": "we will",
    r"\bthey['’]ll\b": "they will",

    # Forms of "have"
    r"\bi['’]ve\b": "i have",
    r"\byou['’]ve\b": "you have",
    r"\bwe['’]ve\b": "we have",
    r"\bthey['’]ve\b": "they have",

    # Forms of "would"
    r"\bi['’]d\b": "i would",
    r"\byou['’]d\b": "you would",
    r"\bhe['’]d\b": "he would",
    r"\bshe['’]d\b": "she would",
    r"\bwe['’]d\b": "we would",
    r"\bthey['’]d\b": "they would",

    # Negative contractions
    r"\bisn['’]t\b": "is not",
    r"\baren['’]t\b": "are not",
    r"\bwasn['’]t\b": "was not",
    r"\bweren['’]t\b": "were not",
    r"\bhasn['’]t\b": "has not",
    r"\bhaven['’]t\b": "have not",
    r"\bhadn['’]t\b": "had not",
    r"\bdon['’]t\b": "do not",
    r"\bdoesn['’]t\b": "does not",
    r"\bdidn['’]t\b": "did not",
    r"\bcan['’]t\b": "cannot",
    r"\bcouldn['’]t\b": "could not",
    r"\bshouldn['’]t\b": "should not",
    r"\bwouldn['’]t\b": "would not",
    r"\bwon['’]t\b": "will not",
    r"\bmustn['’]t\b": "must not",
    r"\bneedn['’]t\b": "need not",

    # Contractions with "there"
    r"\bthere['’]s\b": "there is",
    r"\bthere['’]re\b": "there are",
    r"\bthere['’]ll\b": "there will",
    r"\bthere['’]ve\b": "there have",
    r"\bthere['’]d\b": "there would"
}


cleaned = posts.loc[usable_text, "clean_text"].copy()
for contraction, expansion in contractions.items():
    cleaned = cleaned.str.replace(contraction, expansion, regex=True)

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
lemmatized = [
    " ".join(
        token.text.lower() if token.pos_ == "PROPN" else token.lemma_.lower()
        for token in doc
        if token.is_alpha
    )
    for doc in nlp.pipe(cleaned, batch_size=3000, n_process=NLP_PROCESSES)
]
posts["lemmas"] = pd.Series(lemmatized, index=cleaned.index, dtype="string")
del cleaned, lemmatized
display(posts.loc[usable_text, ["id", "clean_text", "lemmas"]].head(5))

,id,clean_text,lemmas
0,703265,q be ready anons public awakening coming q ret...,q be ready anon public awakening come q retruth
1,807614,enough is enough retruth,enough be enough retruth
5,807621,joe biden wants this video removed from the in...,joe biden want this video remove from the inte...
8,807625,truth soros ukraine,truth soros ukraine
9,703272,what a glorious day that will be,what a glorious day that will be


In [5]:
has_lemmas = posts["lemmas"].fillna("").str.strip().ne("")
eligible = posts.loc[has_lemmas].drop_duplicates("lemmas").index.to_numpy()
train_indices, validation_indices = train_test_split(eligible, test_size=0.20, random_state=SEED)

stop_words = sorted(ENGLISH_STOP_WORDS - {"not", "no", "never", "nor"})
vectorizer = CountVectorizer(
    stop_words=stop_words,
    token_pattern=r"(?u)\b(?=\w*[a-z])\w\w+\b",
    min_df=10,
    max_df=0.90,
)
train_matrix = vectorizer.fit_transform(posts.loc[train_indices, "lemmas"])
validation_matrix = vectorizer.transform(posts.loc[validation_indices, "lemmas"])

train_mask = train_matrix.getnnz(axis=1) >= MIN_KNOWN_TERMS
validation_mask = validation_matrix.getnnz(axis=1) >= MIN_KNOWN_TERMS
train_matrix = train_matrix[train_mask]
validation_matrix = validation_matrix[validation_mask]
words = vectorizer.get_feature_names_out()

print(f"Train: {train_matrix.shape[0]:,} testi")
print(f"Validation: {validation_matrix.shape[0]:,} testi")
print(f"Vocabolario: {len(words):,} termini")

Train: 399,113 testi
Validation: 99,863 testi
Vocabolario: 20,797 termini


In [8]:
try:
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
except ImportError:
    CoherenceModel = None
    print("Gensim non disponibile. La colonna Coherence c_v resterà vuota.")


def top_words(model, limit=10):
    return [words[topic.argsort()[-limit:][::-1]].tolist() for topic in model.components_]


if CoherenceModel is not None:
    coherence_docs = posts.loc[validation_indices[validation_mask], "lemmas"]
    coherence_docs = coherence_docs.sample(min(20_000, len(coherence_docs)), random_state=SEED)
    analyzer = vectorizer.build_analyzer()
    vocabulary = vectorizer.vocabulary_
    tokens = [[term for term in analyzer(text) if term in vocabulary] for text in coherence_docs]
    dictionary = Dictionary(tokens)

models = {}
metrics = []
for k in K_VALUES:
    print(f"Addestramento LDA con K={k}", flush=True)
    model = LatentDirichletAllocation(
        n_components=k,
        learning_method="online",
        learning_decay=0.7,
        batch_size=3000,
        max_iter=5,
        random_state=SEED,
        n_jobs=N_JOBS,
    ).fit(train_matrix)
    models[k] = model
    topics = top_words(model)
    topic_words = [word for topic in topics for word in topic]
    coherence = np.nan
    if CoherenceModel is not None:
        coherence = CoherenceModel(
            topics=topics, texts=tokens, dictionary=dictionary, coherence="c_v", processes=1
        ).get_coherence()
    metrics.append({
        "K": k,
        "Perplexity": model.perplexity(validation_matrix),
        "Coherence c_v": coherence,
        "Topic diversity": len(set(topic_words)) / len(topic_words),
    })

display(pd.DataFrame(metrics).set_index("K").round(4))

Gensim non disponibile. La colonna Coherence c_v resterà vuota.
Addestramento LDA con K=5
Addestramento LDA con K=7
Addestramento LDA con K=8
Addestramento LDA con K=9
Addestramento LDA con K=10
Addestramento LDA con K=12


,Perplexity,Coherence c_v,Topic diversity
K,,,
5,2921.7869,NaN,0.7800
7,2985.0094,NaN,0.8571
8,3046.0454,NaN,0.8375
9,3107.5200,NaN,0.7889
10,3139.4821,NaN,0.8000
12,3229.6561,NaN,0.8167


In [9]:
import json
import subprocess

validation_texts = posts.loc[validation_indices[validation_mask], "lemmas"]
sample = validation_texts.sample(min(20_000, len(validation_texts)), random_state=SEED)

analyzer = vectorizer.build_analyzer()
vocabulary = vectorizer.vocabulary_
tokens = [[word for word in analyzer(text) if word in vocabulary] for text in sample]
topics = [top_words(models[row["K"]]) for row in metrics]

calculate_coherence = """
import json
import sys
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

tokens, model_topics = json.load(sys.stdin)
dictionary = Dictionary(tokens)
scores = [
    CoherenceModel(
        topics=topics, texts=tokens, dictionary=dictionary,
        coherence="c_v", processes=1,
    ).get_coherence()
    for topics in model_topics
]
print(json.dumps(scores))
"""

result = subprocess.run(
    ["/opt/anaconda3/bin/conda", "run", "--no-capture-output",
     "-n", "lds-coherence", "python", "-c", calculate_coherence],
    input=json.dumps([tokens, topics]),
    text=True,
    capture_output=True,
)
if result.returncode:
    raise RuntimeError(result.stderr or result.stdout)

for row, score in zip(metrics, json.loads(result.stdout), strict=True):
    row["Coherence c_v"] = score

display(pd.DataFrame(metrics).set_index("K").round(4))

,Perplexity,Coherence c_v,Topic diversity
K,,,
5,2921.7869,0.6643,0.7800
7,2985.0094,0.6856,0.8571
8,3046.0454,0.6647,0.8375
9,3107.5200,0.6575,0.7889
10,3139.4821,0.6568,0.8000
12,3229.6561,0.6392,0.8167


In [16]:
K_CHOSEN = 7
lda = models[K_CHOSEN]

topic_words = pd.DataFrame({
    "Topic": range(1, K_CHOSEN + 1),
    "Parole principali": [", ".join(terms) for terms in top_words(lda, limit=20)],
})
with pd.option_context("display.max_colwidth", None):
    display(topic_words)

posts["known_terms"] = 0
posts["topic"] = pd.Series(pd.NA, index=posts.index, dtype="Int64")
posts["topic_score"] = np.nan
posts["topic_margin"] = np.nan

rows_to_score = posts.index[has_lemmas]
for start in range(0, len(rows_to_score), 20_000):
    rows = rows_to_score[start:start + 20_000]
    matrix = vectorizer.transform(posts.loc[rows, "lemmas"])
    counts = matrix.getnnz(axis=1)
    posts.loc[rows, "known_terms"] = counts
    supported = counts >= MIN_KNOWN_TERMS
    if supported.any():
        probabilities = lda.transform(matrix[supported])
        first = probabilities.max(axis=1)
        second = np.partition(probabilities, -2, axis=1)[:, -2]
        selected_rows = rows[supported]
        posts.loc[selected_rows, "topic"] = probabilities.argmax(axis=1) + 1
        posts.loc[selected_rows, "topic_score"] = first
        posts.loc[selected_rows, "topic_margin"] = first - second

posts["classification_status"] = "Classificabile"
posts.loc[posts["topic_margin"].lt(MIN_TOPIC_MARGIN), "classification_status"] = "Topic ambiguo"
posts.loc[posts["known_terms"].lt(MIN_KNOWN_TERMS), "classification_status"] = "Pochi termini nel vocabolario"
posts.loc[usable_text & ~has_lemmas, "classification_status"] = "Lemmatizzazione vuota"
posts.loc[~usable_text, "classification_status"] = "Testo vuoto dopo la pulizia"

display(posts["classification_status"].value_counts().rename("Testi distinti").to_frame())

,Topic,Parole principali
0,1,"new, news, year, just, say, covid, biden, report, vaccine, million, use, price, high, pay, gas, break, police, company, death, government"
1,2,"biden, ukraine, russia, war, obama, china, israel, border, putin, joe, hunter, iran, military, treason, terrorist, usa, country, world, democrats, nazi"
2,3,"truth, maga, trump, fjb, usa, saveamerica, xteam, biden, americafirst, trumptruth, trumpwon, thehashtagchat, phpnew, lilypadlounge, covfefe, truthsocial, democrats, news, ncswic, deepstate"
3,4,"god, thank, love, great, good, day, patriot, come, truth, time, follow, make, not, pray, life, family, bless, jesus, happy, today"
4,5,"state, law, election, fbi, court, new, live, judge, texas, watch, border, news, school, great, florida, california, illegal, governor, trump, crime"
5,6,"trump, president, not, vote, biden, america, election, country, democrat, win, party, donald, kamala, maga, joe, need, harris, american, democrats, white"
6,7,"not, people, just, like, know, no, think, say, make, want, need, time, look, right, truth, try, tell, thing, lie, come"


,Testi distinti
classification_status,
Classificabile,402420
Pochi termini nel vocabolario,115942
Topic ambiguo,108977
Testo vuoto dopo la pulizia,54814
Lemmatizzazione vuota,790


In [17]:
def sample_distinct(pool, signatures, seed, limit=10):
    selected = []
    for index, row in pool.sample(frac=1, random_state=seed).iterrows():
        signature = row["clean_text"][:1200]
        if any(SequenceMatcher(None, signature, other).ratio() >= 0.87 for other in signatures):
            continue
        signatures.append(signature)
        selected.append(index)
        if len(selected) == limit:
            break
    return selected


review_columns = ["id", "text", "known_terms", "topic_score", "topic_margin", "classification_status"]
for topic, group in posts.dropna(subset=["topic"]).groupby("topic"):
    low, high = group["topic_margin"].quantile([0.25, 0.75])
    # Il gruppo alto esclude le righe nel quartile basso anche in caso di parità.
    high_pool = group.loc[group["topic_margin"].ge(high) & group["topic_margin"].gt(low)]
    low_pool = group.loc[group["topic_margin"].le(low)]
    signatures = []
    high_rows = sample_distinct(high_pool, signatures, SEED)
    low_rows = sample_distinct(low_pool, signatures, SEED + 1)
    examples = posts.loc[high_rows + low_rows, review_columns].copy()
    examples.insert(0, "Gruppo", ["Margine alto"] * len(high_rows) + ["Margine basso"] * len(low_rows))
    print(f"Topic {topic} — {len(high_rows)} esempi a margine alto, {len(low_rows)} a margine basso")
    with pd.option_context("display.max_colwidth", None):
        display(examples.round({"topic_score": 3, "topic_margin": 3}))

Topic 1 — 10 esempi a margine alto, 10 a margine basso


,Gruppo,id,text,known_terms,topic_score,topic_margin,classification_status
59685,Margine alto,783907,When is the Android app coming?,3,0.785,0.749,Classificabile
362663,Margine alto,360754,They arrived just in time to get their 40 billion,4,0.828,0.799,Classificabile
204417,Margine alto,74621,Method Man Checks Fan Who Photoshopped His Beard Off: 'This Is Suckashit' https://hiphopdx.com/news/id.68010/title.method-man-checks-fan-who-photoshopped-his-beard-off-this-is-suckashit,6,0.722,0.546,Classificabile
331213,Margine alto,302449,RNA for Moderna’s Omicron Booster Manufactured by CIA-Linked Company https://unlimitedhangout.com/2022/08/investigative-reports/rna-for-modernas-omicron-booster-manufactured-by-cia-linked-company/,8,0.785,0.649,Classificabile
657,Margine alto,703947,Who needs a Wall when we have a wall with teeth? The Rio Grande is warm enough and there’s plenty of illegal food crossing every day (loaded with HIV/AIDS).,14,0.726,0.579,Classificabile
41852,Margine alto,758780,WATCH: Shocking footage shows Pelosi and Schumer reacting to Jan. 6 riot in real time https://www.washingtonexaminer.com/politics/shocking-video-congress-leaders-jan6-riot,10,0.716,0.497,Classificabile
396380,Margine alto,437863,Those who were given a pass on their debt means that forgiveness of debt is a taxable event. Fact check.,7,0.662,0.403,Classificabile
81207,Margine alto,841979,<emoji: red_circle> Top Biden Adviser Forced to Recuse Herself From Consulting on Key Issues After Revealed Ties https://resistthemainstream.org/top-biden-adviser-forced-to-recuse-herself-from-consulting-on-key-issues-after-revealed-ties/?utm_source=truthsocial,9,0.803,0.678,Classificabile
519231,Margine alto,801121,"Imagine saying you don't control gas prices for an entire year of price increases, and then taking credit for the decline <emoji: clown_face><emoji: clown_face><emoji: clown_face><emoji: clown_face>",11,0.685,0.424,Classificabile
145653,Margine alto,7845,Education Department offers a second chance to 7.5 million borrowers who defaulted on student loans — here's who is eligible https://www.cnbc.com/2022/08/19/education-dept-offers-student-loan-borrowers-in-default-a-second-chance.html,11,0.623,0.399,Classificabile


Topic 2 — 10 esempi a margine alto, 10 a margine basso


,Gruppo,id,text,known_terms,topic_score,topic_margin,classification_status
655873,Margine alto,2360791,#Truth #Trump2024 #Biden #bidencrimefamily #Democrats #republicans #China #russia #Ukraine #NATO #corruption #dojcorruption #FBIcorruption #MoneyLaundering #OBAMAGATE #Obama #constitution #Insurrection #covid #vaccine #trumpderangementsyndrome #FreeTrump #MAGA #lgbtq #FJB #AmericaFirst #Jan6 #WETHEPEOPLE #usa #TrumpWon #AmericaFirst #SaveAmerica #riggedelection #WWG1WGA #UK #racism #LiberalismIsAMentalDisorder #Zelensky #NoMoreMoneyToUkraine #defundtheUN #DefundDOJ #desantis #Nazi #Israel #israelunderattack #terrorists #Islam #Muslims #Gaza #westbank #IDF #evil #jews #Israeli #istandwithisrael #Hamas #Hezbollah #antisemitism #Islamophobia #hostages #ceasefire #canada #christianity #EpsteinClientList #HunterBiden #Houthis #desantis #deepstate #border #trump #closetheborder #Islamist #Guns #bidenomics #Iran #TrumpAssasinationAttempt #war #CIA #oct7 #kamalaharris #BorderCzar #faith #TrumpVance2024 #TamponTim #elonmusk #DNC #deepstate #RFKJr #debate #SecretService #RyanRouth #Stopgap,83,0.836,0.680,Classificabile
56432,Margine alto,779245,"If Ukraine and Russia reach a ceasefire or negotiate an end to the conflict shortly, do we, the American taxpayers, get our 14 billion dollars back?",12,0.934,0.923,Classificabile
665939,Margine alto,2394948,RT @realDonaldTrump“LAWFARE IS DEAD”,3,0.784,0.748,Classificabile
645429,Margine alto,2210037,. . . . . #taketheredpill #truth #arizonapatriots #americafirst #arizonafirst #takebackamerica #maga #maga2024 #trumpsupporter #az #ultramaga #trumpsupporters #trump2024 #scottsdale #scottsdaleaz #scottsdalearizona #carefree #carefreeaz #carefreearizona #cavecreek #cavecreekaz #cavecreekarizona #phoenix #phoenixaz #arizona #lgbfjb #awakenotwoke #takebackamerica2022 #saveamerica #2A #2AShallNotBeInfringed,25,0.644,0.380,Classificabile
604057,Margine alto,1813380,#IlhanOmarIsAPaidAgent for #Iran #IncestOmar #PaidAgent #IlhanOmar #Qatar and IRAN https://links.truthsocial.com/link/110515469132713877,3,0.829,0.800,Classificabile
262682,Margine alto,177062,FJB and the Deep State he rode in on.,4,0.613,0.370,Classificabile
306527,Margine alto,252558,#BioLabs #DOD #Truth #Ukraine #BioWeapons @MarshaRWBlue @DevinNunes @Kash Part 1,5,0.857,0.832,Classificabile
286180,Margine alto,217279,Taliban outlaw the production of poppies in Afghanistan | Just The News https://justthenews.com/world/taliban-outlaw-production-poppies-afghanistan,7,0.632,0.353,Classificabile
272931,Margine alto,192147,"The ONE thing Putin asked was for Ukraine to pledge it would never join NATO. If Ukraine had simply done that, Russian troops would have withdrawn and the war would have ended. (Peace Agreement proposed on April 9th) What did Ukraine do? Joined NATO.",18,0.494,0.333,Classificabile
594134,Margine alto,1712581,The Snake Poem ... .#IlhanOmar #HouseOfRepresentatives #ForeignAffairsCommittee #antisemitic #mapoli #mepolitics #omar #dc #speaker #SOH #talib #antiZionist #Palestinians #occupying #Israel #Jews #TheSquad #RashidaTlaib #Tlaib #AOC #obama #Minnesota NYC #Islam #Hamas #Hezbollah #Iran #Syria #Sudan #Solmalia #antisemitism #ExpelOmar #expelIlhan,27,0.754,0.694,Classificabile


Topic 3 — 10 esempi a margine alto, 10 a margine basso


Gruppo       id  \
575522   Margine alto  1510421   
641398   Margine alto  2154655   
590358   Margine alto  1675378   
551487   Margine alto  1236741   
631748   Margine alto  2057113   
570109   Margine alto  1465520   
617117   Margine alto  1952052   
636485   Margine alto  2103433   
613912   Margine alto  1922026   
552703   Margine alto  1252067   
474683  Margine basso   582934   
463294  Margine basso   564886   
583179  Margine basso  1592746   
598949  Margine basso  1762443   
631200  Margine basso  2055159   
362394  Margine basso   360259   
562192  Margine basso  1376105   
613677  Margine basso  1920125   
642119  Margine basso  2164491   
590570  Margine basso  1678091   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

... (4 outputs left out)

Topic 6 — 10 esempi a margine alto, 10 a margine basso


,Gruppo,id,text,known_terms,topic_score,topic_margin,classification_status
470746,Margine alto,576339,𝐀𝐦𝐞𝐫𝐢𝐜𝐚 𝐖𝐢𝐥𝐥 𝐁𝐞 𝐔𝐧𝐢𝐟𝐢𝐞𝐝 𝐀𝐠𝐚𝐢𝐧. 𝐅𝐮𝐭𝐮𝐫𝐞 𝐏𝐫𝐨𝐯𝐞𝐬 𝐏𝐚𝐬𝐭.,5,0.857,0.833,Classificabile
321592,Margine alto,282182,<emoji: studio_microphone> Joe Rogan on President Trump,4,0.828,0.800,Classificabile
483926,Margine alto,596733,I miss u Trump soo muuch,3,0.785,0.749,Classificabile
13906,Margine alto,719244,A poll conducted after the election found that 17 percent of Biden voters would not have voted for him if they had known about the “Laptop from Hell.” https://hannity.com/media-room/now-available-stream-my-son-hunter-the-story-you-werent-allowed-to-see/,11,0.651,0.481,Classificabile
604743,Margine alto,1822101,Remember? And Laundering LaCivita #FireWiles #FireLaCivita #TraitorWiles #TraitorLaCivita,7,0.766,0.621,Classificabile
409743,Margine alto,465027,"That's right of course! Those machines have to go, however we all need to go vote in every election. Not just the big ones, but the local also! Every vote is very very important! We need to overwhelm the Dems. All states need to turn red!",16,0.877,0.789,Classificabile
600106,Margine alto,1774396,<emoji: rotating_light>ATTENTION TRUE AMERICA<emoji: rotating_light><emoji: rotating_light> <emoji: rotating_light><emoji: rotating_light> FIRST TRUMP LOYALISTS <emoji: rotating_light><emoji: rotating_light> NEOCON NEVERTRUMPERS HIJACKING TRUMPS CAMPAIGN ARE LYING TO TRUMP AND US!! THEY ARE ACTIVELY SABOTAGING TRUMP! THEY DONT BELIEVE THE ELECTION IS STOLEN AND REFUSE TO DEPLOY ELECTION INTEGRITY TEAMS TO SWINGS STATES TO PREVENT IT!!! THESE SWAMP CREATURES NEED TO GO!!! #FireLaCivita #FireWiles #TraitorLaCivita #TraitorWiles #AmericaFirst,31,0.976,0.972,Classificabile
283595,Margine alto,212602,https://gettr.com/post/px76pdbcf9 We’re not his people either… We are “ We The People ” and President Trump is our President !!!!,4,0.877,0.857,Classificabile
296963,Margine alto,233161,Screw that knob. Let the swamp go fight for him.,4,0.828,0.799,Classificabile
398803,Margine alto,441327,Another corrupt pick by Democrats. .,3,0.786,0.750,Classificabile


Topic 7 — 10 esempi a margine alto, 10 a margine basso


,Gruppo,id,text,known_terms,topic_score,topic_margin,classification_status
431688,Margine alto,505517,13 users follow this troll? https://truthsocial.com/@LiqaMadiq,3,0.786,0.750,Classificabile
285672,Margine alto,216269,If that isn’t the most fake cry BS I’ve ever seen! I hope people follow you everywhere! Karma’s a bitch!,8,0.792,0.663,Classificabile
362874,Margine alto,361250,I’ve tried but I can’t follow anyone!!!!!,3,0.786,0.750,Classificabile
9462,Margine alto,713269,Yes I agree I’ve seen a few just muted and blocked them pretty easy to spot after you see them for a bit not difficult,11,0.830,0.719,Classificabile
268110,Margine alto,185180,the joke is liberals trying to appear smart,5,0.857,0.833,Classificabile
447492,Margine alto,538577,"They do it to keep making it look like he did something wrong, when they are showing themselves as corrupt",5,0.857,0.833,Classificabile
216528,Margine alto,91970,Liars don’t know how to tell the truth-I wonder when she first started lying?,8,0.905,0.889,Classificabile
22844,Margine alto,731316,Nothing makes sense with these people. It’s like letting kids run the refinery,8,0.791,0.662,Classificabile
90252,Margine alto,859252,"You might be right. Me personally, I don’t follow the man and Frankly couldn’t give a rats ass about the guy. He’s not running for office. ￼ But not everyone views the world the way you do!! Not everyone is a Christian some people don’t believe in religion and some people are pragmatists￼￼!",18,0.798,0.677,Classificabile
399640,Margine alto,442515,I can’t wait to learn what the real number is!,5,0.857,0.833,Classificabile


In [18]:
TOPIC_LABELS = {
    1: (
        "General News and Commentary",
        "News and commentary on current events in health, the economy, "
        "technology, entertainment, and public affairs.",
    ),
    2: (
        "International Affairs and U.S. Foreign Policy",
        "Discussion of international conflicts, diplomacy, alliances, "
        "and the role of the United States abroad.",
    ),
    3: (
        "MAGA and Political Hashtags",
        "Posts characterized by MAGA-related slogans and political "
        "or community hashtags used on Truth Social.",
    ),
    4: (
        "Faith and Community Support",
        "Posts expressing faith, prayer, encouragement, solidarity, "
        "or greetings within the community.",
    ),
    5: (
        "U.S. Domestic Politics and Institutions",
        "Discussion of U.S. legislation, courts, public officials, "
        "and state or congressional races and endorsements.",
    ),
    6: (
        "U.S. Presidential Campaigns and Voting",
        "Discussion of U.S. presidential candidates, campaign strategy, "
        "polling, and voter mobilization.",
    ),
    7: (
        "General Reactions and Discussion",
        "Conversational posts such as replies, reactions, disagreements, "
        "and interactions between users.",
    ),
}

In [20]:
expected_topics = set(range(1, K_CHOSEN + 1))

if set(TOPIC_LABELS) != expected_topics:
    missing_labels = sorted(expected_topics - TOPIC_LABELS.keys())
    print(f"Add a name and definition before exporting. Unlabeled topics: {missing_labels}")
else:
    definitions = pd.DataFrame.from_dict(
        TOPIC_LABELS,
        orient="index",
        columns=["Category", "Definition"],
    ).sort_index()
    definitions.index.name = "Topic"
    assert definitions.apply(lambda column: column.str.strip().ne("").all()).all()

    names = definitions["Category"].str.strip()

    # The previous cell stores this status in Italian.
    candidates = posts["classification_status"].eq("Classificabile")
    posts["Category"] = posts["topic"].map(names).where(candidates).fillna("Unclassified")
    categories_by_text = posts.set_index("text")["Category"]

    status_names = {
        "Classificabile": "Classified",
        "Topic ambiguo": "Ambiguous topic",
        "Pochi termini nel vocabolario": "Too few terms in the vocabulary",
        "Lemmatizzazione vuota": "Empty after lemmatization",
        "Testo vuoto dopo la pulizia": "Empty after cleaning",
    }
    status_by_text = posts.set_index("text")["classification_status"].replace(status_names)
    rejected_texts = posts.loc[candidates & posts["Category"].eq("Unclassified"), "text"]
    status_by_text.loc[rejected_texts] = "Topic excluded from categorization"

    final = pd.DataFrame({
        "IntegratedPostID": truths["id"],
        "Category": truths["text"].map(categories_by_text).fillna("Unclassified"),
    })

    final_status = truths["text"].map(status_by_text)
    final_status = final_status.mask(missing_text, "Missing text")
    final_status = final_status.mask(blank_text, "Whitespace-only text")

    assert len(final) == len(truths) and final["IntegratedPostID"].is_unique
    assert final_status.notna().all()

    display(definitions)
    display(final_status.value_counts().rename_axis("Classification status").to_frame("Posts"))
    display(final["Category"].value_counts().to_frame("Posts"))

    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    final.to_csv(OUTPUT_FILE, index=False)
    print(f"Exported {len(final):,} posts to {OUTPUT_FILE}.")

,Category,Definition
Topic,,
1,General News and Commentary,News and commentary on current events in healt...
2,International Affairs and U.S. Foreign Policy,"Discussion of international conflicts, diploma..."
3,MAGA and Political Hashtags,Posts characterized by MAGA-related slogans an...
4,Faith and Community Support,"Posts expressing faith, prayer, encouragement,..."
5,U.S. Domestic Politics and Institutions,"Discussion of U.S. legislation, courts, public..."
6,U.S. Presidential Campaigns and Voting,"Discussion of U.S. presidential candidates, ca..."
7,General Reactions and Discussion,"Conversational posts such as replies, reaction..."


,Posts
Classification status,
Classified,514249
Too few terms in the vocabulary,171597
Ambiguous topic,142798
Empty after cleaning,103546
Missing text,11767
Empty after lemmatization,2419


,Posts
Category,
Unclassified,432127
General Reactions and Discussion,175457
Faith and Community Support,78605
U.S. Presidential Campaigns and Voting,69480
General News and Commentary,69144
U.S. Domestic Politics and Institutions,55585
MAGA and Political Hashtags,47416
International Affairs and U.S. Foreign Policy,18562


Exported 946,376 posts to /Users/lorenzobuccolini/Desktop/MSc in Data Science and Business Informatics (UniPi)/Laboratory of Data Science/project /code/Truth-Social-Analysis-and-Data-Warehousing---Decision-Support-Systems-/02_assignment/output/post_categories.csv.
